In [ ]:
from pathlib import Path

import numpy as np
import numpy.typing as npt

from climate_attitudes.visualisation import configure_mpl
from ising import Ising

configure_mpl(Path("../fonts/"))

np.set_printoptions(linewidth=200)

RANDOM_SEED = 202607161539
rng = np.random.default_rng(RANDOM_SEED)

In [ ]:
def get_transition_matrix(model: Ising) -> npt.NDArray[np.float64]:
    N = model.size
    Y0 = (2 * ((np.arange(1 << N)[:, None] >> np.arange(N)) & 1) - 1).astype(np.float64)
    X = np.ones(Y0.shape[0])

    heff = model.parallel_glauber_theta_batch(Y0, X, model.h, model.j, model.adj)
    sum_log2cosh_heff = np.log(2 * np.cosh(heff)).sum(axis=-1)
    s_dot_heff = heff @ Y0.T
    p_transition = np.exp(s_dot_heff - sum_log2cosh_heff)
    return p_transition

In [ ]:
def get_all_transition_matrices(
    params: npt.NDArray[np.float64],
) -> npt.NDArray[np.float64]:
    N = 8
    R = np.empty((params.shape[0], 2**N, 2**N), dtype=np.float64)
    for repeat in range(params.shape[0]):
        h, j = Ising.unpack_params(params[repeat], k=0)
        model = Ising(
            field=h,
            coupling=j,
            infer_structure=True,
            rng=np.random.default_rng(RANDOM_SEED),
        )
        R[repeat] = get_transition_matrix(model)

In [ ]:
params = np.load("../reports/thesis/results/data/model/all_interventions/ising_25.npz")[
    "params"
]
R = get_all_transition_matrices(params)